### Human-in-the-loop

https://docs.langchain.com/oss/python/langchain/middleware/built-in#human-in-the-loop

In [31]:
import os
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [32]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="gpt-4.1",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [33]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "test-approve"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=f"send email to john@test.com with subject 'hello' and body 'how are you?'"
            )
        ]
    },
    config=config,
)

In [34]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='5aa9f495-0b75-416a-a1bb-b94aa8e06131'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_aae4b5f0e2', 'id': 'chatcmpl-DK6P7v32eIxs2iXJl0rJ1lDDgn7Cg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf7c7-1f28-7441-a5bc-af53ef0440af-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'hello', 'body': 'how are you?'}, 'id': 'call_K8Q94mAH287SrBl

In [35]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! waiting for approval....")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! waiting for approval....
Result: The email has been sent to john@test.com with the subject "hello" and the body "how are you?". If you need to send another email or have any other requests, let me know!


#### Reject

In [36]:
config = {"configurable": {"thread_id": "test-reject"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=f"send email to john@test.com with subject 'hello' and body 'how are you?'"
            )
        ]
    },
    config=config,
)

In [37]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! waiting for approval....")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "reject"}
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! waiting for approval....
Result: It looks like the email wasn't sent. Would you like to proceed with sending the email to john@test.com, or is there anything you want to change before I send it?


#### Edit

In [43]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"


def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


agent = create_agent(
    model="gpt-4.1",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [44]:
config = {"configurable": {"thread_id": "test-edit"}}

result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=f"send email to john@test.com with subject 'hello' and body 'how are you?'"
            )
        ]
    },
    config=config,
)

In [45]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='237b0a24-eb20-4dcd-86d2-7612b8171b3f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_ed3c347d43', 'id': 'chatcmpl-DK6Q7ii9fpx64U7ROrivmSv502Ubn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf7c8-103b-7993-bcb7-fae3bc6e511f-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'hello', 'body': 'how are you?'}, 'id': 'call_EhaQBlChuYodopF

In [47]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("Paused! waiting for approval....")

    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action": {
                            "name": "send_email_tool",
                            "args": {
                                "recipient": "correct",
                                "subject": "corrected subject",
                                "body": "edited by human before sending",
                            },
                        },
                    }
                ]
            }
        ),
        config=config,
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! waiting for approval....
Result: The email has been sent to correct with the subject "corrected subject" and the message "edited by human before sending".

However, it seems you requested to send the email to john@test.com with the subject "hello" and body "how are you?" Would you like me to send the email with your original details instead? Please confirm or provide any desired changes.


In [48]:
result

{'messages': [HumanMessage(content="send email to john@test.com with subject 'hello' and body 'how are you?'", additional_kwargs={}, response_metadata={}, id='237b0a24-eb20-4dcd-86d2-7612b8171b3f'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 97, 'total_tokens': 125, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-2025-04-14', 'system_fingerprint': 'fp_ed3c347d43', 'id': 'chatcmpl-DK6Q7ii9fpx64U7ROrivmSv502Ubn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cf7c8-103b-7993-bcb7-fae3bc6e511f-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recepient': 'correct', 'subject': 'corrected subject', 'body': 'edited by human before sending'}, 'id